# Notebook for batch processes

## OpenAI

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

### Check the Status of a Batch

In [ ]:
batch_id = "batch_abc123"
batch = client.batches.retrieve(batch_id)
print(batch)

In [ ]:
# List the 10 most recent batches
client.batches.list(limit=10)

### Retrieve Results

Once the batch is complete, you can download the output by making a request against the Files API via the `output_file_id` field from the Batch object and writing it to a file on your machine, in this case `batch_output.jsonl`

In [ ]:
import json

def get_batch_results(batch_id: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch = client.batches.retrieve(batch_id)

    if batch.status != "completed":
        return {
            "status": batch.status,
            "message": "Batch not completed yet."
        }

    output_file_id = batch.output_file_id
    output_file = client.files.content(output_file_id)

    results = {}
    for line in output_file.iter_lines():
        record = json.loads(line)
        custom_id = record["custom_id"]
        output_text = record["response"]["output_text"]
        results[custom_id] = output_text

    return results

In [ ]:
results = get_batch_results(batch_id)

model_name = "gpt-5.1"

# save to local file
local_filename = f"outputs/responses/{model_name}/batch_question_responses.jsonl"
with open(local_filename, "w") as f:
    for custom_id, output_text in results.items():
        record = {
            "custom_id": custom_id,
            "output_text": output_text
        }
        f.write(json.dumps(record) + "\n")

### Cancel a Batch Job

If necessary, can cancel an ongoing batch. The batch's status will change to `cancelling` until in-flight requests are complete (up to 10 minutes), after which the status will change to `cancelled`.

In [ ]:
batch_id = "batch_abc123"
client.batches.cancel(batch_id)

## Anthropic's Claude

In [ ]:
import anthropic
from dotenv import load_dotenv
import os

load_dotenv(override=True)
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

### Monitor Batch Job Status

In [ ]:
# Automatically fetches more pages as needed.
for message_batch in client.messages.batches.list(limit=20):
    print(message_batch)

In [ ]:
import time

MESSAGE_BATCH_ID = "your_message_batch_id_here"

message_batch = None
while True:
    message_batch = client.messages.batches.retrieve(
        MESSAGE_BATCH_ID
    )
    if message_batch.processing_status == "ended":
        break

    print(f"Batch {MESSAGE_BATCH_ID} is still processing...")
    time.sleep(60)
print(message_batch)

### Retrieve Results

In [ ]:
MESSAGE_BATCH_ID = "your_message_batch_id_here"

# Stream results file in memory-efficient chunks, processing one at a time
for result in client.messages.batches.results(MESSAGE_BATCH_ID,):
    match result.result.type:
        case "succeeded":
            print(f"Success! {result.custom_id}")
        case "errored":
            if result.result.error.type == "invalid_request":
                # Request body must be fixed before re-sending request
                print(f"Validation error {result.custom_id}")
            else:
                # Request can be retried directly
                print(f"Server error {result.custom_id}")
        case "expired":
            print(f"Request expired {result.custom_id}")

In [ ]:
import json

def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    MESSAGE_BATCH_ID = "your_message_batch_id_here"

    results = {}
    for result in client.messages.batches.results(MESSAGE_BATCH_ID,):
        custom_id = result.custom_id
        record = json.loads(result)
        custom_id = output_text = record["result"]["content"][0]["text"]
        results[custom_id] = output_text
    return results

results = get_batch_results(batch_id)

model_name = "claude_4.5_sonnet"

# save to local file
local_filename = f"outputs/responses/{model_name}/batch_question_responses.jsonl"
with open(local_filename, "w") as f:
    for custom_id, output_text in results.items():
        record = {
            "custom_id": custom_id,
            "output_text": output_text
        }
        f.write(json.dumps(record) + "\n")

### Cancel a Batch Job

In [ ]:
MESSAGE_BATCH_ID = "your_message_batch_id_here"

message_batch = client.messages.batches.cancel(
    MESSAGE_BATCH_ID,
)
print(message_batch)

## Google's Gemini

In [ ]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=gemini_api_key)

### Monitor Batch Job Status

In [ ]:
import time

job_name = "batch_job_from_file.name"

print(f"Polling status for job: {job_name}")

# Poll the job status until it's completed.
while True:
    batch_job = client.batches.get(name=job_name)
    if batch_job.state.name in ('JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
        break
    print(f"Job not finished. Current state: {batch_job.state.name}. Waiting 30 seconds...")
    time.sleep(30)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

In [ ]:
job_name = "batch_job_from_file.name"

print(f"Retrieving status for job: {job_name}")

batch_job = client.batches.get(name=job_name)
print(f"Current state: {batch_job.state.name}")
print(f"Job details: {batch_job}")

### Retrieve Results

Once a file-based job succeeds, the results are written to an output file in the [Files API](./File_API.ipynb).

In [ ]:
if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    # The result file is also a JSONL file. Parse and print each line.
    for line in file_content.splitlines():
      if line:
        parsed_response = json.loads(line)
        # Pretty-print the JSON for readability
        print(json.dumps(parsed_response, indent=2))
        print("-" * 20)
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")

In [ ]:
import json

def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch_job = client.batches.get(name=batch_name)

    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    results = {}
    for line in file_content.splitlines():
        if line:
            record = json.loads(line)
            custom_id = record["key"]
            output_text = record["candidates"][0]["content"]["parts"]["text"]
            results[custom_id] = output_text
    return results

results = get_batch_results(batch_id)

model_name = "gemmini-2.5-flash"

# save to local file
local_filename = f"outputs/responses/{model_name}/batch_question_responses.jsonl"
with open(local_filename, "w") as f:
    for custom_id, output_text in results.items():
        record = {
            "custom_id": custom_id,
            "output_text": output_text
        }
        f.write(json.dumps(record) + "\n")

### Cancel a Batch Job

In [ ]:
# try:
#   job_to_cancel_name = "batches/your-job-name-here" # <-- REPLACE THIS
#   print(f"Attempting to cancel job: {job_to_cancel_name}")
#   client.batches.cancel(name=job_to_cancel_name)
#   print("Job cancellation request sent.")
# except Exception as e:
#   print(f"Error cancelling job: {e}")